# Chapter 9 — limma-voom Differential Expression (R)

**Prerequisite:** run `pseudobulk_limma_voom.ipynb` first to generate the `pseudobulk/` CSVs.

This notebook runs the same limma-voom pipeline as Chapter 3, now at two resolutions:

1. **Global** — 18 cell types from the global h5ad (`pseudobulk/*.csv`)
2. **Subtype** — author-annotated subtypes per compartment (`pseudobulk/subtypes/*/`)

The cross-reference table (Step 5) shows which Chapter 3 bulk markers localise
to which cell types — the single-cell resolution answer to the bulk DE signal.

## Step 1 — Load packages

In [ ]:
suppressPackageStartupMessages({
  library(limma)
  library(edgeR)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(readr)
  library(purrr)
})

cat('limma', as.character(packageVersion('limma')), '\n')
cat('edgeR', as.character(packageVersion('edgeR')), '\n')
cat('ggplot2', as.character(packageVersion('ggplot2')), '\n')

## Step 2 — Configuration

Update `CTRL_VALUE` and `CASE_VALUE` to match what you saw in Step 3 of the Python notebook.

In [ ]:
CTRL_VALUE  <- 'Ctrl'
CASE_VALUE  <- 'EuE'
FDR_CUT     <- 0.05
FC_CUT      <- 1.0    # log2FC threshold for volcano

# Chapter 3 bulk DE markers — cross-validation targets
CH3_MARKERS <- c(
  'MMP2', 'MMP9', 'MMP11', 'MMP14',
  'POSTN', 'ACTA2', 'FN1', 'VIM',
  'HOXA10', 'HOXA11',
  'VEGFA', 'IL6',
  'PGR', 'ESR1'
)

dir.create('results', showWarnings = FALSE)

## Step 3 — limma-voom helper function

In [ ]:
run_limma_voom <- function(counts_csv, meta_csv, ctrl, case) {
  counts <- read_csv(counts_csv, show_col_types = FALSE)
  genes  <- counts[[1]]
  counts <- as.matrix(counts[, -1])
  rownames(counts) <- genes

  meta   <- read_csv(meta_csv, show_col_types = FALSE)

  # Align sample order
  keep   <- meta$condition %in% c(ctrl, case)
  meta   <- meta[keep, ]
  counts <- counts[, meta$sample, drop = FALSE]

  if (sum(meta$condition == ctrl) < 2 || sum(meta$condition == case) < 2) {
    return(NULL)   # not enough replication
  }

  condition <- factor(meta$condition, levels = c(ctrl, case))
  design    <- model.matrix(~ condition)

  dge  <- DGEList(counts = counts)
  keep <- filterByExpr(dge, design)
  dge  <- dge[keep, , keep.lib.sizes = FALSE]
  dge  <- calcNormFactors(dge)

  v    <- voom(dge, design, plot = FALSE)
  fit  <- lmFit(v, design)
  fit  <- eBayes(fit)
  res  <- topTable(fit, coef = 2, number = Inf, sort.by = 'none')
  res$gene <- rownames(res)
  res
}

cat('limma-voom function ready\n')

## Step 4 — Run DE: global cell-type analysis (cell types with enough donors in both groups)

Reads from `pseudobulk/*.csv` (exported by the Python notebook).

In [ ]:
counts_files <- list.files('pseudobulk', pattern = '_counts\\.csv$', full.names = TRUE)
# exclude subtypes subdirectory
counts_files <- counts_files[!grepl('subtypes', counts_files)]

global_results <- list()

for (cf in counts_files) {
  mf <- sub('_counts\\.csv', '_meta.csv', cf)
  ct <- sub('.*pseudobulk/(.*?)_counts\\.csv', '\\1', cf)
  ct <- gsub('_', ' ', ct)

  res <- tryCatch(
    run_limma_voom(cf, mf, CTRL_VALUE, CASE_VALUE),
    error = function(e) { cat('  FAILED', ct, ':', conditionMessage(e), '\n'); NULL }
  )

  if (!is.null(res)) {
    res$celltype <- ct
    global_results[[ct]] <- res
    n_sig <- sum(res$adj.P.Val < FDR_CUT, na.rm = TRUE)
    cat(sprintf('  %-30s  %5d DE genes (FDR<0.05)\n', ct, n_sig))
  }
}

cat('\nCompleted:', length(global_results), 'cell types\n')

## Step 5 — Volcano plots: global cell types

Chapter 3 bulk markers are highlighted in colour.
One panel per cell type, saved as PDF.

In [ ]:
make_volcano <- function(res, title, ch3_markers, fdr_cut = 0.05, fc_cut = 1.0) {
  res <- res %>%
    mutate(
      neg_log10_fdr = -log10(pmax(adj.P.Val, 1e-300)),
      sig  = adj.P.Val < fdr_cut & abs(logFC) > fc_cut,
      direction = case_when(
        sig & logFC >  fc_cut ~ 'up',
        sig & logFC < -fc_cut ~ 'down',
        TRUE                  ~ 'ns'
      ),
      is_ch3 = gene %in% ch3_markers
    )

  n_up   <- sum(res$direction == 'up')
  n_down <- sum(res$direction == 'down')

  ggplot(res, aes(x = logFC, y = neg_log10_fdr)) +
    geom_point(aes(colour = direction), size = 0.8, alpha = 0.5) +
    geom_point(data = filter(res, is_ch3),
               aes(colour = direction), size = 3, shape = 18) +
    ggrepel::geom_text_repel(
      data = filter(res, is_ch3 & sig),
      aes(label = gene), size = 2.5, max.overlaps = 20
    ) +
    geom_hline(yintercept = -log10(fdr_cut), linetype = 'dashed', colour = 'grey50') +
    geom_vline(xintercept = c(-fc_cut, fc_cut), linetype = 'dashed', colour = 'grey50') +
    scale_colour_manual(values = c('up' = '#e74c3c', 'down' = '#3498db', 'ns' = '#cccccc')) +
    labs(
      title    = sprintf('%s\n\u2191%d \u2193%d', title, n_up, n_down),
      x        = expression(log[2]*FC~~(EuE~vs~Ctrl)),
      y        = expression(-log[10]*~FDR),
      colour   = NULL
    ) +
    theme_bw(base_size = 9) +
    theme(legend.position = 'none', plot.title = element_text(size = 8))
}

# Install ggrepel if needed
if (!requireNamespace('ggrepel', quietly = TRUE)) install.packages('ggrepel')

# Build plot list
plots <- imap(global_results, ~ make_volcano(.x, .y, CH3_MARKERS))

# Arrange in grid
n     <- length(plots)
ncols <- 4
nrows <- ceiling(n / ncols)

if (!requireNamespace('patchwork', quietly = TRUE)) install.packages('patchwork')
library(patchwork)

p_grid <- wrap_plots(plots, ncol = ncols) +
  plot_annotation(
    title    = 'Pseudo-bulk DE: EuE vs Ctrl — cell types with enough donors',
    subtitle = 'Diamonds = Chapter 3 bulk DE markers',
    theme    = theme(plot.title = element_text(size = 12))
  )

ggsave('results/volcano_global_18celltypes.pdf',
       p_grid, width = ncols * 4, height = nrows * 3.5, limitsize = FALSE)

cat('Saved: results/volcano_global_18celltypes.pdf\n')
p_grid

## Step 6 — Cross-reference: Chapter 3 markers × cell types

For each Chapter 3 bulk DE marker, show in which cell type it is significantly DE
and at what fold change — the cell-type-resolved version of the bulk result.

In [ ]:
all_de <- bind_rows(global_results)

ch3_summary <- all_de %>%
  filter(gene %in% CH3_MARKERS) %>%
  select(gene, celltype, logFC, adj.P.Val) %>%
  mutate(
    logFC = round(logFC, 2),
    sig   = ifelse(adj.P.Val < FDR_CUT, '*', '')
  ) %>%
  pivot_wider(id_cols = gene, names_from = celltype,
              values_from = logFC)

print(ch3_summary, n = Inf, width = Inf)

write_csv(ch3_summary, 'results/ch3_markers_by_celltype.csv')
cat('Saved: results/ch3_markers_by_celltype.csv\n')

## Step 7 — Run DE: subtype analysis (stromal + lymphocyte)

Reads from `pseudobulk/subtypes/stromal/` and `pseudobulk/subtypes/lymphocyte/`.
Same limma-voom function, finer resolution.

In [ ]:
run_compartment_de <- function(compartment) {
  dir_path <- file.path('pseudobulk', 'subtypes', compartment)
  if (!dir.exists(dir_path)) {
    cat('Directory not found:', dir_path, '\n')
    return(list())
  }

  counts_files <- list.files(dir_path, pattern = '_counts\\.csv$', full.names = TRUE)
  results <- list()

  cat('\n──', compartment, '──\n')
  for (cf in counts_files) {
    mf <- sub('_counts\\.csv', '_meta.csv', cf)
    ct <- basename(sub('_counts\\.csv', '', cf))
    ct <- gsub('_', ' ', ct)

    res <- tryCatch(
      run_limma_voom(cf, mf, CTRL_VALUE, CASE_VALUE),
      error = function(e) { cat('  FAILED', ct, ':', conditionMessage(e), '\n'); NULL }
    )

    if (!is.null(res)) {
      res$celltype    <- ct
      res$compartment <- compartment
      results[[ct]]   <- res
      n_sig <- sum(res$adj.P.Val < FDR_CUT, na.rm = TRUE)
      cat(sprintf('  %-30s  %5d DE genes (FDR<0.05)\n', ct, n_sig))
    }
  }
  results
}

stromal_results    <- run_compartment_de('stromal')
lymphocyte_results <- run_compartment_de('lymphocyte')

## Step 8 — Volcano plots: subtype compartments

In [ ]:
plot_compartment <- function(results, compartment_name, outfile) {
  if (length(results) == 0) {
    cat('No results for', compartment_name, '\n')
    return(invisible(NULL))
  }
  plots  <- imap(results, ~ make_volcano(.x, .y, CH3_MARKERS))
  ncols  <- min(4, length(plots))
  nrows  <- ceiling(length(plots) / ncols)
  p_grid <- wrap_plots(plots, ncol = ncols) +
    plot_annotation(
      title    = paste('Pseudo-bulk DE: EuE vs Ctrl —', compartment_name, 'subtypes'),
      subtitle = 'Diamonds = Chapter 3 bulk DE markers',
      theme    = theme(plot.title = element_text(size = 12))
    )
  ggsave(outfile, p_grid,
         width = ncols * 4, height = nrows * 3.5, limitsize = FALSE)
  cat('Saved:', outfile, '\n')
  p_grid
}

plot_compartment(stromal_results,    'stromal',    'results/volcano_stromal_subtypes.pdf')
plot_compartment(lymphocyte_results, 'lymphocyte', 'results/volcano_lymphocyte_subtypes.pdf')

## Step 9 — Save all DE results

In [ ]:
# Global results
all_global <- bind_rows(global_results)
write_csv(all_global, 'results/de_global_all_celltypes.csv')

# Subtype results
all_subtype <- bind_rows(c(stromal_results, lymphocyte_results))
if (nrow(all_subtype) > 0) {
  write_csv(all_subtype, 'results/de_subtypes_all.csv')
}

cat('Saved: results/de_global_all_celltypes.csv\n')
cat('Saved: results/de_subtypes_all.csv\n')

cat('\nDE genes (FDR<0.05) per cell type:\n')
all_global %>%
  filter(adj.P.Val < FDR_CUT) %>%
  count(celltype, name = 'n_DE') %>%
  arrange(desc(n_DE)) %>%
  as.data.frame() %>%
  print()

---
## What you just did — and why it matters

**Three resolutions, one limma-voom:**

| Chapter | Resolution | n samples | Question |
|---------|-----------|-----------|----------|
| 3 | Bulk tissue | 24 | Is anything DE between EuE and Ctrl? |
| 9 (global) | 13 of 18 cell types | per-celltype | Which cell type carries the bulk signal? |
| 9 (subtype) | fibroblast/NK/T subtypes | per-subtype | Which exact subpopulation drives it? |

The cross-reference table (Step 6) is the scientific payoff:
if MMP genes are DE specifically in fibroblast subtypes, the ECM remodeling
story is localised. If VEGFA is DE in endothelial cells, the angiogenesis
signal converges across bulk, NSForest, and pseudo-bulk — three independent lines
of evidence from one dataset.

> *The bulk result told you something was different.*  
> *The 18-type result told you where.*  
> *The subtype result told you exactly which cells.*